In [4]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
import matplotlib.pyplot as plt

# データセットのロード
data = load_iris()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

# トレーニングとテストデータの分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# KFoldの設定
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# パラメータの設定
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 4, 5]
}

# GridSearchCVの設定
grid_search = GridSearchCV(XGBRegressor(random_state=42), param_grid, cv=kf, scoring='neg_mean_absolute_error', n_jobs=-1)

# GridSearchCVの実行
grid_search.fit(X_train, y_train)

# ベストパラメータとスコアの表示
print(f'Best parameters: {grid_search.best_params_}')
print(f'Best MAE: {-grid_search.best_score_}')

# ベストモデルでKFoldを使って予測
best_model = grid_search.best_estimator_
kf_predictions = np.zeros(len(y))

# KFoldを使って学習と予測
for train_index, test_index in kf.split(X):
    X_train_fold, X_test_fold = X.iloc[train_index], X.iloc[test_index]
    y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]

    # ベストモデルで学習
    best_model.fit(X_train_fold, y_train_fold)
    
    # テストデータに対する予測
    y_pred = best_model.predict(X_test_fold)
    
    # 全体の予測に対する平均を計算するために各Foldの予測を蓄積
    kf_predictions[test_index] = y_pred

# 各Foldの予測値の平均を最終予測値として計算
final_predictions = kf_predictions

# モデルの性能評価
mae = mean_absolute_error(y, final_predictions)
print(f'Mean Absolute Error: {mae}')

# 学習曲線のプロット
def plot_learning_curve(estimator, title, X, y, cv, scoring='neg_mean_absolute_error'):
    train_sizes, train_scores, val_scores = learning_curve(
        estimator, X, y, cv=cv, scoring=scoring, n_jobs=-1, 
        train_sizes=np.linspace(0.1, 1.0, 10)
    )

    train_scores_mean = -np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    val_scores_mean = -np.mean(val_scores, axis=1)
    val_scores_std = np.std(val_scores, axis=1)

    plt.figure()
    plt.title(title)
    plt.xlabel("Training examples")
    plt.ylabel("Mean Absolute Error")
    plt.ylim(0, 10)
    plt.grid()

    plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.1, color="r")
    plt.fill_between(train_sizes, val_scores_mean - val_scores_std,
                     val_scores_mean + val_scores_std, alpha=0.1, color="g")
    plt.plot(train_sizes, train_scores_mean, 'o-', color="r", label="Training error")
    plt.plot(train_sizes, val_scores_mean, 'o-', color="g", label="Cross-validation error")

    plt.legend(loc="best")
    return plt

# ベストモデルの学習曲線のプロット
plot_learning_curve(best_model, "Learning Curve (XGBRegressor)", X, y, cv=kf)
plt.show()


Best parameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
Best MAE: 0.08335148449720389
Mean Absolute Error: 0.06004451709128261


NameError: name 'learning_curve' is not defined